## 1. Compute equilibria of the following growth model

Compute equilibria of the following growth model (which is similar to HW6 but now the
stochastic terms are just accounting "wedges"):

$$\max_{\{c_t, x_t, \ell_t\}} E \sum_{t=0}^{\infty} \beta^t \left\{ \frac{(c_t \ell_t^\psi)^{1-\sigma}}{1-\sigma} \right\} N_t$$

subj. to

\begin{gather*}
c_t + (1 + \tau_{xt}) x_t = r_t k_t + (1 - \tau_{ht}) w_t h_t + \kappa_t \\
N_{t+1} k_{t+1} = \bigl[(1-\delta) k_t + x_t\bigr] N_t \\
S_t = P S_{t-1} + Q \epsilon_t, \quad S_t = [\log z_t,\, \tau_{ht},\, \tau_{xt},\, \log g_t] \\
h_t + \ell_t = 1 \\
N_t = (1+\gamma_n)^t \\
c_t, x_t \geq 0 \quad \text{in all states.}
\end{gather*}

 (a) Use data for the United States to infer the wedges.

(b) Input all wedges and show the model and data match exactly.

(c) Input the wedges one at a time and interpret the results.


### **1.1 Data Source**

In this part we rebuild the business-cycle-accounting panel from primary U.S. sources. All series are quarterly and cover **1948Q1 – 2024Q4**. Real variables are obtained by dividing by the GDP deflator.

$-$ **BEA NIPA Table 1.1.5 — Gross Domestic Product**
- GDP: Gross domestic product, market prices
- pce: Personal consumption expenditures
- private_investment: Gross private domestic investment
- net_exports: Net exports of goods and services 
- govt_consumption_and_investment: Government consumption expenditures and gross investment
- durable_goods, nondurable_goods: sub-lines of PCE goods

$-$ **BEA NIPA Table 1.1.6 — Real GDP, Chained Dollars**
- gdp_real: Real GDP (millions of chained (2017) dollars, SAAR)

$-$ **BEA NIPA Table 3.10.5 — Government Consumption Expenditures and General Government Gross Output**
- government_consumption: Government consumption expenditures
- government_investment $\equiv$ govt_consumption_and_investment − government_consumption


$-$ **BEA NIPA Tables 3.3 + 3.2 — Taxes on goods and services**
- sales_taxes (Table 3.3, state & local): *Sales taxes*
- excise_taxes_sl (Table 3.3, state & local): *Excise taxes*
- excise_taxes_federal (Table 3.2, federal): *Excise taxes*
- $\tau_c$: effective consumption-tax rate,  
  $$\tau_c \;=\; \dfrac{\text{sales\_taxes} + \text{excise\_taxes\_sl} + \text{excise\_taxes\_federal}}{\mathrm{GDP}}$$

$-$ **BLS**
- total_worked_hours: Hours worked for all workers in US.

$-$ **OECD Historical Population**
- iP: Population 15–64, persons. Annual observations 1950–2023 are extended with the mean year-on-year growth rate to **1947–2025** and then interpolated to quarterly frequency with cubic splines.


### **1.2 Per-capita accounting variables**

&emsp;&emsp;Following CKM (2007), we transform the quarterly panel into real per-capita series suitable for business-cycle accounting. 


- **Per capita output** $(y)$: real GDP $-$ sales taxes $+$ services from consumer durables (return = 4%) $+$ depreciation of durables (annualized rate 25%), deflated by the GDP deflator and divided by population 15–64.

- **Per capita hours** $(h)$: aggregate hours worked (BLS Total U.S. economy) divided by population 15–64. 
- **Per capita investment** $(x)$: gross investment $+$ personal consumption expenditures on durables net of sales taxes, all deflated by the GDP deflator and divided by population 15–64.

- **Per capita government consumption** $(g)$: government final consumption expenditures $+$ net exports of goods and services, all deflated by the GDP deflator and divided by population 15–64.



In [32]:
# --- Libraries and data paths (BEA NIPA + BLS + OECD historical population) -----------
include(joinpath("function", "Data_clean.jl"))
using CSV, DataFrames

# BEA NIPA tables + BLS total-economy hours + OECD 15-64 population
BEA115_PATH  = raw"data/Table 1.1.5. Gross Domestic Product.xlsx"
BEA116_PATH  = raw"data/Table 1.1.6. Real Gross Domestic Product, Chained Dollars.xlsx"
BEA33_PATH   = raw"data/Table 3.3. State and Local Government Current Receipts and Expenditures.xlsx"
BEA32_PATH   = raw"data/Table 3.2. Federal Government Current Receipts and Expenditures.xlsx"
BEA3105_PATH = raw"data/Table 3.10.5. Government Consumption Expenditures and General Government Gross Output.xlsx"
HOURS_PATH   = raw"data/total-economy-hours-employment.xlsx"
POP_PATH     = raw"data/population.xlsx"

# --- Load the BEA (+BLS +OECD-pop) quarterly panel -------------------------------------
# bea_wide : BEA 1.1.5 + 1.1.6 (→ PGDP, 2009=1) + 3.3 + 3.2 + 3.10.5 + BLS hours + iP
# panel    : HW7 1948Q1-2024Q4 restriction with the final 10 columns used downstream.
bea_wide = read_bea_panel(
    path_115 = BEA115_PATH, path_116 = BEA116_PATH,
    path_33  = BEA33_PATH,  path_32  = BEA32_PATH, path_3105 = BEA3105_PATH,
    path_hours_worked = HOURS_PATH, path_population = POP_PATH,
    deflator_base_year = 2009,
)

panel = build_us_quarterly_panel(
    path_115 = BEA115_PATH, path_116 = BEA116_PATH,
    path_33  = BEA33_PATH,  path_32  = BEA32_PATH, path_3105 = BEA3105_PATH,
    path_hours_worked = HOURS_PATH, path_population = POP_PATH,
    year_start = 1948, year_end = 2024, deflator_base_year = 2009,
    fill_tauc_missing = :quarterly_mean,
)

Row,time_label,GDP,PGDP,gross_investment,government_consumption,net_exports,iP,total_worked_hours,durable_goods,tauc
,String,Float64,Float64,Float64,Float64,Float64,Float64?,Float64?,Float64,Float64
1,Q1-1948,265742.0,0.133987,54739.0,33338.0,7293.0,9.70349e7,124.156,23547.0,0.0384153
2,Q2-1948,272567.0,0.135194,58582.0,34638.0,5205.0,9.72917e7,124.436,24019.0,0.0383707
3,Q3-1948,279196.0,0.137691,61295.0,35881.0,4949.0,9.75532e7,125.816,25277.0,0.0383765
4,Q4-1948,280366.0,0.138111,60700.0,37237.0,4501.0,9.78227e7,125.353,24971.0,0.0382991
5,Q1-1949,275034.0,0.137376,53392.0,38333.0,6478.0,9.81009e7,123.908,24436.0,0.0384153
6,Q2-1949,271351.0,0.135998,47536.0,39086.0,6283.0,9.83777e7,123.063,26389.0,0.0383707
7,Q3-1949,272889.0,0.135369,51473.0,38432.0,5179.0,9.86406e7,121.305,27313.0,0.0383765
8,Q4-1949,270627.0,0.135381,49257.0,38121.0,2999.0,9.88771e7,120.113,28356.0,0.0382991
9,Q1-1950,280828.0,0.13517,57429.0,38276.0,2203.0,9.90786e7,120.487,29335.0,0.0384153


In [33]:
# --- Per-capita BCA wedge-accounting series from the BEA panel -------------------------
# CKM (2007) conventions: r_d = 4% annual, δ_d = 25% annual, K^{CD}_0 = 16 × real durables.
# Outputs are in real USD per 15-64 person (SAAR); h in hours per person per year.
data_pc = percapita_bea_ckm_from_hw7_panel(panel;
    r_d_annual      = 0.04,
    delta_d_annual  = 0.25,
    kcd_init_factor = 16.0,
)

data_pc

Row,time_label,y,h,x,g,iP,tauc,KCD_real
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,Q1-1948,21950.8,1279.5,5951.79,3125.13,9.70349e7,0.0384153,2.81187e12
2,Q2-1948,22201.9,1279.0,6209.83,3029.14,9.72917e7,0.0383707,2.79248e12
3,Q3-1948,22243.3,1289.72,6372.91,3039.72,9.75532e7,0.0383765,2.77636e12
4,Q4-1948,22198.9,1281.43,6270.32,3089.32,9.78227e7,0.0382991,2.76727e12
5,Q1-1949,21850.6,1263.07,5705.36,3325.08,9.81009e7,0.0384153,2.75604e12
6,Q2-1949,21712.7,1250.92,5449.69,3391.01,9.83777e7,0.0383707,2.74266e12
7,Q3-1949,21858.8,1229.77,5821.79,3266.03,9.86406e7,0.0383765,2.74637e12
8,Q4-1949,21652.8,1214.77,5716.89,3071.84,9.88771e7,0.0382991,2.75755e12
9,Q1-1950,22383.7,1216.07,6394.42,3022.52,9.90786e7,0.0384153,2.77564e12


In [56]:
# 测试：CKM 基准参数 + 示例楔子（σ = 1.001）
# 运行前请将工作目录设为 HW7（与 `function/` 同级），或改 `include` 路径。

include(joinpath("function", "BCA_steady_state.jl"))

# --- 深度参数（截图 / 论文表格；β、δ 为年率，与 mleq 一致）---
psi   = 2.24
sigma = 1.001      # 你指定；论文常记 σ = 1
beta  = 0.9722
theta = 0.35
delta = 0.0464

# 年人口增长 1.5%、年技术增长 1.6% → 季度 gn, gz（与 sr328/mleq.m 相同换算）
gn = (1.0 + 0.015)^(1/4) - 1.0
gz = (1.0 + 0.016)^(1/4) - 1.0

# --- 示例楔子水平（可与 mleq 默认无条件均值对比）：z=1, τ_l=0.05, τ_x=0, ĝ≈0.07 ---
z      = 1.0
tau_l  = 0.05
tau_x  = 0.0
g      = exp(log(0.07))

ss = bca_steady_state(; z=z, tau_l=tau_l, tau_x=tau_x, g=g,
                      gn=gn, gz=gz, beta=beta, delta=delta,
                      psi=psi, sigma=sigma, theta=theta)

println("── BCA detrended steady state (test) ──")
println("  β̂ = ", round(ss.beta_hat, digits=6))
println("  k̂, ĉ, l, ŷ, x̂ = ", round(ss.k, digits=5), ", ", round(ss.c, digits=5), ", ",
      round(ss.l, digits=5), ", ", round(ss.y, digits=5), ", ", round(ss.x, digits=5))
println("  kl = k̂/l = ", round(ss.kl, digits=5))
println("  ξ₁, ξ₂, ξ₃ = ", round(ss.xi1, digits=5), ", ", round(ss.xi2, digits=5), ", ", round(ss.xi3, digits=5))

# 也可用 Sbar 向量接口
Sbar  = [log(z), tau_l, tau_x, log(g)]
param = [gn, gz, beta, delta, psi, sigma, theta]
ss2 = bca_steady_state_from_Sbar(Sbar, param)
println("\nmax|ss - ss2| = ", maximum(abs.([ss.k-ss2.k, ss.c-ss2.c, ss.l-ss2.l, ss.y-ss2.y, ss.x-ss2.x])))




── BCA detrended steady state (test) ──
  β̂ = 0.968346
  k̂, ĉ, l, ŷ, x̂ = 2.92156, 0.43207, 0.29638, 0.66018, 0.15812
  kl = k̂/l = 9.85763
  ξ₁, ξ₂, ξ₃ = 0.17185, 0.61406, 0.06229

max|ss - ss2| = 0.0


### **2. Solve the log-linearized policy function**

If we substitute for $\hat{c}_t$ using the resource constraint and then log-linearize the conditions around the steady state, we get:

\begin{align*}
0 &= E_t \{ a_1 \tilde{k}_t + a_2 \tilde{k}_{t+1} + a_3 \tilde{h}_t + a_4 \tilde{z}_t + a_5 \tilde{\tau}_{ht} + a_6 \tilde{g}_t \}\\
0 &= E_t \{ b_1 \tilde{k}_t + b_2 \tilde{k}_{t+1} + b_3 \tilde{k}_{t+2} + b_4 \tilde{h}_t + b_5 \tilde{h}_{t+1} + b_6 \tilde{z}_t + b_7 \tilde{\tau}_{xt} + b_8 \tilde{g}_t + b_9 \tilde{z}_{t+1} + b_{10} \tilde{\tau}_{xt+1} + b_{11} \tilde{g}_{t+1} \}
\end{align*}


where $\tilde{k}_t = \log \hat{k}_t/\hat{k}_{ss}  $,  $\tilde{h}_t = \log h_t / h_{ss}$, $\tilde{z}_t = \log z_t / z_{ss}$ and $\tilde{g}_t = \log \hat{g}_t / \hat{g}_{ss}$. Labor and investment wedges are usually taken as level deviations * $\tilde{\tau}_{ht} = \tau_{ht} - \tau_{h,ss}$,$\quad \tilde{\tau}_{xt} = \tau_{xt} - \tau_{x,ss}$


These equations can be stacked up as follows 
$$0 = E_t \left\{ \underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 0 & 0 \\ 0 & b_3 & b_5 \end{bmatrix}}_{=A_1} \begin{bmatrix} \tilde{k}_{t+1} \\ \tilde{k}_{t+2} \\ \tilde{h}_{t+1} \end{bmatrix} + \underbrace{\begin{bmatrix} 0 & -1 & 0 \\ a_1 & a_2 & a_3 \\ b_1 & b_2 & b_4 \end{bmatrix}}_{=A_2} \begin{bmatrix} \tilde{k}_t \\ \tilde{k}_{t+1} \\ \tilde{h}_t \end{bmatrix} + \underbrace{\begin{bmatrix} 0 & 0 & 0 & 0 \\ a_4 & a_5 & 0 & a_6 \\ b_6 & 0 & b_7 & b_8 \end{bmatrix}}_{=B_1} S_t + \underbrace{\begin{bmatrix} 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ b_9 & 0 & b_{10} & b_{11} \end{bmatrix}}_{=B_2} S_{t+1} \right\}$$


For this problem, we are looking for a solution of the form:
\begin{align*}
\tilde{k}_{t+1} &= A \tilde{k}_t + B S_t\\
Z_t &= C \tilde k_t + D S_t \\
S_t &= P S_{t-1} + Q \epsilon_t 
\end{align*}


where $Z_t = [\tilde{k}_{t+1}, \tilde{h}_t]'$ and $S_t$ are the stochastic exogenous variables.


Applying Vaughan, we compute the *generalized eigenvalues and eigenvectors* for matrices $A_1$ and $-A_2$ because $A_1$ is not invertible.  Let $V$ be the eigenvectors and $\Lambda$ is a diagonal matrix with eigenvalues, so that

$$A_2 V = -A_1 V \Lambda.$$

Sort the eigenvalues in $\Lambda$ and the associated columns in the matrix of eigenvectors $V$ so that the eigenvalue inside the unit circle is in the (1,1) position of $\Lambda$. Then,

\begin{align*}
A &= V_{11}\,\Lambda(1,1)\,V_{11}^{-1}, \\
C &= V_{21}\,V_{11}^{-1}.
\end{align*}

Once we have $A$ and $C$, we plug them into the system above—along with $S_{t+1} = P S_t + Q \epsilon_{t+1}$—and we can form a linear system in $B$ and $D$. Since the policy function for the first element of $Z_t$ is the same as $A, B$, we'll just consider the policy rule for $\tilde{h}_t$, namely:

$$\tilde{h}_t = C_2 \tilde{k}_t + D_2 S_t$$

Plugging the policy rules into the first-order equations yields the following:

$$0 = (a_1 + a_2 A + a_3 C_2)\, \tilde{k}_t + \bigl(a_2 B + a_3 D_2 + \begin{bmatrix} a_4 & a_5 & 0 & a_6 \end{bmatrix}\bigr) S_t$$

$$0 = (b_1 + b_2 A + b_3 A^2 + b_4 C_2 + b_5 C_2 A)\, \tilde{k}_t + \bigl(b_2 B + b_3 A B + b_3 B P + b_4 D_2 + b_5 C_2 B + \boxed{b_5 D_2 P} + \begin{bmatrix} b_6 & 0 & b_7 & b_8 \end{bmatrix} + \begin{bmatrix} b_9 & 0 & b_{10} & b_{11} \end{bmatrix} P\bigr) S_t$$

We can check that the coefficients on $\tilde{k}_t$ are zero at the solution to calculate $C_2$. (Notice that they do not depend on $B$ or $D_2$.) That leaves eight unknowns, namely four elements of $B$ and four elements of $D_2$ and eight linear equation, namely the coefficients on $S_t$, which must be set equal to zero at the solution:

$$0 = a_2 B + a_3 D_2 + \begin{bmatrix} a_4 & a_5 & 0 & a_6 \end{bmatrix}$$

$$0 = b_2 B + b_3 A B + b_3 B P + b_4 D_2 + b_5 C_2 B + b_5 D_2 P + \begin{bmatrix} b_6 & 0 & b_7 & b_8 \end{bmatrix} + \begin{bmatrix} b_9 & 0 & b_{10} & b_{11} \end{bmatrix} P$$
We just have to stack these eight equations and solve the linear system. To do this, we'll need to use the fact that $\operatorname{vec}(FGH) = (H' \otimes F)\,\operatorname{vec}(G)$ for any matrices $F$, $G$, and $H$.


One last thing to note. If we set $\tau_{ht} = 0$, $\tau_{xt} = 0$, and $g_t = 0$, we are back to the simple case of Homework 1. Thus, we have a test case for the codes.

In [57]:
using LinearAlgebra
include(joinpath("function", "BCA_policy_linear.jl"))

# Quarterly calibration, same as mleq.m: deep params + wedge steady levels + AR(1) matrix P
param_q = [
    (1.015)^(1 / 4) - 1,          # gn
    (1.016)^(1 / 4) - 1,          # gz
    0.9722^(1 / 4),               # beta (quarterly)
    1 - (1 - 0.0464)^(1 / 4),     # delta (quarterly)
    2.24, 1.001, 0.35,            # psi, sigma, theta
]
Sbar_q = [log(1.0), 0.05, 0.0, log(0.07)]   # [log z, τ_h, τ_x, log g] steady levels
P_q    = Matrix(0.995 * I(4))
P0_q   = (I(4) - P_q) * Sbar_q

# Route ①: mleq quadratic root + γ; Route ②: Vaughan + 8×8 linear system
m      = policy_mleq_style(Sbar_q, P_q, P0_q, param_q)
A1, A2 = build_A1_A2_from_dR(m.Z, param_q, m.dR)
pol    = policy_vaughan_linear_policy(A1, A2, P_q, m.Z, param_q, m.dR)

gk = Float64(real(m.gammak))
println("Route ①  γ_k = ", gk, "   γ = ", Vector{Float64}(real.(m.gamma)))
println("Route ②  A   = ", pol.A, "   C₂ = ", pol.C2)
println("         B   = ", vec(pol.B))
println("         D₂  = ", vec(pol.D2))
println("|γ_k − A|    = ", abs(gk - pol.A))
println("Route ② max|res_S| (intratemp, Euler) = ",
        maximum(abs.(pol.res_S_intra)), ", ", maximum(abs.(pol.res_S_euler)))

Route ①  γ_k = 0.9678648060817299   γ = [0.03570673876161733, -0.02635354300368871, -0.06237911116725793, 0.0011415551713267783]
Route ②  A   = 0.9678648060817304   C₂ = 1.666949505087286e-9
         B   = [0.03570673876161781, -0.026353543003689063, -0.06237911116725876, 0.0011415551713267935]
         D₂  = [2.8286534535094905e-11, 5.114028327955279e-11, -4.941613105809284e-11, 1.8094759456463778e-10]
|γ_k − A|    = 5.551115123125783e-16
Route ② max|res_S| (intratemp, Euler) = 6.462348535570529e-27, 0.0


### **A.2.2.1 test (Appendix / HW1 limit)**

Under $g_n = g_z = \psi \to 0$, $\sigma = \delta = 1$, $\tau_h = \tau_x = 0$, $z = 1$, the closed-form solution is $\gamma_k = A = \theta$ and $B = [\,1-\theta,\ 0,\ -1+\beta\theta,\ 0\,]$ on $S_t = [\log z,\ \tau_h,\ \tau_x,\ \log g]^{\prime}$, which both routes should match up to the finite-difference error from the small $\psi$ floor.


In [58]:
using LinearAlgebra
include(joinpath("function", "BCA_policy_linear.jl"))

# Tiny ψ keeps the labor FOC non-degenerate; other params match Appendix A.2.2.1.
r221 = run_appendix_a221_test(; beta = 0.96, theta = 0.35, rho_z = 0.9, psi_floor = 1e-7)

println("\nmax|Route ① − analytical| on (A, B) = ", max(r221.err_route1_A, r221.err_route1_B))
println("max|Route ② − analytical| on (A, B) = ", max(r221.err_route2_A, r221.err_route2_B))


=== Appendix A.2.2.1 — analytical vs numerical (ψ = 1.0e-7) ===
  Analytical A = θ = 0.35
  Route ① γ_k = 0.3499999999706343  |Δ| = 2.936567655709155e-11
  Route ② A   = 0.34999999997063425  |Δ| = 2.936573206824278e-11
  Analytical B (z, τ_h, τ_x, log g) = [0.65, 0.0, -0.664, 0.0]
  Route ① γ   = [0.6499999959838855, -7.074107012884413e-8, -0.6640000245990761, -1.7724494077435364e-12]  max|Δ| = 7.074107012884413e-8
  Route ② B   = [0.6499999959838854, -7.074107012884411e-8, -0.664000024599076, -1.772449407743536e-12]  max|Δ| = 7.074107012884411e-8

max|Route ① − analytical| on (A, B) = 7.074107012884413e-8
max|Route ② − analytical| on (A, B) = 7.074107012884411e-8
